In [1]:
# ============================================================
# pilot4_expB_settingA1_step4 setup
# Upload:
#   1. config.py
# Read from Google Drive:
#   2. Step 3 zip
# This cell prepares files and directories for Step4T.
# ============================================================

import os
import sys
import shutil
import zipfile
import importlib
from pathlib import Path
from google.colab import files
from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# 1. Create project structure
# ------------------------------------------------------------

PILOT_ROOT = "/content/pilot_4"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")
DATA_DIR = os.path.join(PILOT_ROOT, "data")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Make pilot_4 importable as a package.
with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# ------------------------------------------------------------
# 2. Upload config.py
# ------------------------------------------------------------

print("Upload config.py")
uploaded_config = files.upload()

config_upload_name = None

for name in uploaded_config.keys():
    if name.startswith("config") and name.endswith(".py"):
        config_upload_name = name
        break

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

config_target_path = os.path.join(PILOT_ROOT, "config.py")

shutil.copy(
    os.path.join("/content", config_upload_name),
    config_target_path,
)

# ------------------------------------------------------------
# 3. Load config and prepare Step4T directories
# ------------------------------------------------------------

import pilot_4.config as cfg
importlib.reload(cfg)

# Step4T input/output dirs from config
os.makedirs(cfg.STEP4T_INPUT_DIR, exist_ok=True)

# Clear old Step4T input files to avoid mixing stale Step3 files.
for old_json in Path(cfg.STEP4T_INPUT_DIR).glob("*_experiment_A_text_*.json"):
    old_json.unlink()

# Clear old Step4T outputs to avoid stale files in the final result.
if os.path.exists(cfg.STEP4T_OUTPUT_DIR):
    shutil.rmtree(cfg.STEP4T_OUTPUT_DIR)

os.makedirs(cfg.STEP4T_OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 4. Use Step3 zip from Google Drive
# ------------------------------------------------------------

step3_zip_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step3/p4_settingA_step3.zip"
)

print("\nUsing Step3 zip from Google Drive:")
print(step3_zip_path)

if not step3_zip_path.exists():
    raise FileNotFoundError(f"Step3 zip not found: {step3_zip_path}")

# ------------------------------------------------------------
# 5. Unzip Step3 output
# ------------------------------------------------------------

tmp_extract_dir = "/content/_step3_extract_tmp"

if os.path.exists(tmp_extract_dir):
    shutil.rmtree(tmp_extract_dir)

os.makedirs(tmp_extract_dir, exist_ok=True)

with zipfile.ZipFile(str(step3_zip_path), "r") as zf:
    zf.extractall(tmp_extract_dir)

# Copy Step3 scene json files into cfg.STEP4T_INPUT_DIR.
json_files = sorted(Path(tmp_extract_dir).rglob(cfg.STEP4T_INPUT_PATTERN))

if not json_files:
    raise FileNotFoundError(
        "No *_experiment_A_text_*.json files found in the Step3 zip."
    )

copied_json_files = []

for jf in json_files:
    target_path = os.path.join(cfg.STEP4T_INPUT_DIR, jf.name)
    shutil.copy(str(jf), target_path)
    copied_json_files.append(jf.name)

# ------------------------------------------------------------
# 6. Setup summary
# ------------------------------------------------------------

print("\nStep4 setup complete.")
print("Uploaded config:", config_upload_name)
print("Config copied to:", config_target_path)
print("Step3 zip from Drive:", step3_zip_path)
print("Step3 json files copied to:", cfg.STEP4T_INPUT_DIR)
print("Number of Step3 json files copied:", len(copied_json_files))
print("Step4T output dir:", cfg.STEP4T_OUTPUT_DIR)
print("Step4T experiment:", cfg.STEP4T_EXPERIMENT_NAME)
print("Allowed labels:", sorted(cfg.STEP4T_ALLOWED_LABELS))

Mounted at /content/drive
Upload config.py


Saving config_p4_expB_settingA1.py to config_p4_expB_settingA1.py

Using Step3 zip from Google Drive:
/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step3/p4_settingA_step3.zip

Step4 setup complete.
Uploaded config: config_p4_expB_settingA1.py
Config copied to: /content/pilot_4/config.py
Step3 zip from Drive: /content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step3/p4_settingA_step3.zip
Step3 json files copied to: /content/pilot_4/data/step3A_diverse_text_preserve_text_direction
Number of Step3 json files copied: 6
Step4T output dir: /content/pilot_4/data/p4_expB_settingA1_lr
Step4T experiment: Step4_expB_settingA1_implicit_transitive_lr
Allowed labels: ['left_of', 'right_of']


In [2]:
import json
import re
import hashlib
from collections import Counter
import pandas as pd

# Map cfg variables into the global names used by later cells.
RANDOM_SEED = cfg.RANDOM_SEED

INVERSE_RELATION_MAP = cfg.INVERSE_RELATION_MAP
INV = dict(INVERSE_RELATION_MAP)

STEP4T_EXPERIMENT_NAME = cfg.STEP4T_EXPERIMENT_NAME
STEP4T_INPUT_DIR = cfg.STEP4T_INPUT_DIR
STEP4T_INPUT_PATTERN = cfg.STEP4T_INPUT_PATTERN
STEP4T_OUTPUT_DIR = cfg.STEP4T_OUTPUT_DIR
STEP4T_COMBINED_OUTPUT_FILE = cfg.STEP4T_COMBINED_OUTPUT_FILE
STEP4T_VALID_SPANS_OUTPUT_FILE = cfg.STEP4T_VALID_SPANS_OUTPUT_FILE
STEP4T_SUMMARY_FILE = cfg.STEP4T_SUMMARY_FILE
STEP4T_INDEX_CSV_FILE = cfg.STEP4T_INDEX_CSV_FILE

STEP4T_CANDIDATE_EVIDENCE_TYPE = cfg.STEP4T_CANDIDATE_EVIDENCE_TYPE
IMPLICIT_TRANSITIVE = cfg.IMPLICIT_TRANSITIVE

STEP4T_ALLOWED_LABELS = cfg.STEP4T_ALLOWED_LABELS
ALLOWED_LABELS = set(STEP4T_ALLOWED_LABELS)

STEP4T_EXCLUDE_EXPLICIT_DIRECT = cfg.STEP4T_EXCLUDE_EXPLICIT_DIRECT
STEP4T_EXCLUDE_EXPLICIT_INVERSE = cfg.STEP4T_EXCLUDE_EXPLICIT_INVERSE

STEP4T_SUPPORT_RULES = cfg.STEP4T_SUPPORT_RULES
STEP4T_MIN_SUPPORT_EDGES = cfg.STEP4T_MIN_SUPPORT_EDGES
STEP4T_SAVE_ALL_SUPPORTING_PATHS = cfg.STEP4T_SAVE_ALL_SUPPORTING_PATHS
STEP4T_KEEP_PRIMARY_SUPPORT_PATH = cfg.STEP4T_KEEP_PRIMARY_SUPPORT_PATH

STEP4T_SPAN_SELECTION_MODE = cfg.STEP4T_SPAN_SELECTION_MODE
STEP4T_SAVE_VALID_SPANS_FILE = cfg.STEP4T_SAVE_VALID_SPANS_FILE
STEP4T_REQUIRE_VALID_PROBE_SPANS_FOR_MAIN_OUTPUT = cfg.STEP4T_REQUIRE_VALID_PROBE_SPANS_FOR_MAIN_OUTPUT

STEP4T_LABEL_SOURCE = cfg.STEP4T_LABEL_SOURCE
STEP4T_SUPPORTING_RELATION_SOURCE = cfg.STEP4T_SUPPORTING_RELATION_SOURCE
STEP4T_SAMPLE_SOURCE = cfg.STEP4T_SAMPLE_SOURCE
STEP4T_TEXT_SOURCE = cfg.STEP4T_TEXT_SOURCE

STEP4T_PRINT_FILE_SUMMARY = cfg.STEP4T_PRINT_FILE_SUMMARY
STEP4T_PRINT_GLOBAL_SUMMARY = cfg.STEP4T_PRINT_GLOBAL_SUMMARY
STEP4T_PRINT_EXAMPLES = cfg.STEP4T_PRINT_EXAMPLES
STEP4T_NUM_EXAMPLES_TO_PRINT = cfg.STEP4T_NUM_EXAMPLES_TO_PRINT
STEP4T_SAVE_INDEX_CSV = cfg.STEP4T_SAVE_INDEX_CSV
STEP4T_DEFINITION = cfg.STEP4T_DEFINITION

STEP3_DIR = Path(STEP4T_INPUT_DIR)
OUT_DIR = Path(STEP4T_OUTPUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Config bridge loaded.")
print("Experiment:", STEP4T_EXPERIMENT_NAME)
print("STEP3_DIR:", STEP3_DIR)
print("OUT_DIR:", OUT_DIR)
print("Input pattern:", STEP4T_INPUT_PATTERN)
print("Allowed labels:", sorted(ALLOWED_LABELS))
print("Candidate evidence type:", STEP4T_CANDIDATE_EVIDENCE_TYPE)
print("New evidence type:", IMPLICIT_TRANSITIVE)

Config bridge loaded.
Experiment: Step4_expB_settingA1_implicit_transitive_lr
STEP3_DIR: /content/pilot_4/data/step3A_diverse_text_preserve_text_direction
OUT_DIR: /content/pilot_4/data/p4_expB_settingA1_lr
Input pattern: FloorPlan*_experiment_A_text_diverse.json
Allowed labels: ['left_of', 'right_of']
Candidate evidence type: implicit_labeled
New evidence type: implicit_transitive


In [3]:
# ============================================================
# Cell 2. Load Step3 files
# ============================================================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


step3_files = sorted(STEP3_DIR.glob(STEP4T_INPUT_PATTERN))

if not step3_files:
    raise FileNotFoundError(
        f"No Step3 files found under {STEP3_DIR} with pattern {STEP4T_INPUT_PATTERN}"
    )

print(f"Found {len(step3_files)} Step3 files:")
for p in step3_files:
    print(" -", p.name)

step3_data_by_file = {}

for path in step3_files:
    data = load_json(path)
    step3_data_by_file[path.name] = data

print("\nLoaded files:", len(step3_data_by_file))

Found 6 Step3 files:
 - FloorPlan1_experiment_A_text_diverse.json
 - FloorPlan2_experiment_A_text_diverse.json
 - FloorPlan3_experiment_A_text_diverse.json
 - FloorPlan4_experiment_A_text_diverse.json
 - FloorPlan5_experiment_A_text_diverse.json
 - FloorPlan6_experiment_A_text_diverse.json

Loaded files: 6


In [4]:
# ============================================================
# Cell 3. General helper functions
# ============================================================

def stable_hash(text, n=20):
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:n]


def find_all_spans(text, mention):
    """
    Find all exact mention spans in paragraph text.
    """
    if not mention:
        return []
    pattern = re.escape(mention)
    return [{"start": m.start(), "end": m.end()} for m in re.finditer(pattern, text)]


def build_object_index(paragraph):
    """
    Build uid -> object metadata from paragraph["object_mentions"].
    """
    obj_index = {}
    for obj in paragraph.get("object_mentions", []):
        uid = obj.get("object_uid")
        if uid:
            obj_index[uid] = obj
    return obj_index


def get_obj_meta(uid, obj_index, fallback_pair_target=None, role=None):
    """
    Return object metadata for uid.
    fallback_pair_target is used if uid is not found in object_mentions.
    """
    if uid in obj_index:
        obj = obj_index[uid]
        return {
            "uid": obj.get("object_uid", uid),
            "id": obj.get("object_id"),
            "type": obj.get("object_type"),
            "mention_text": obj.get("mention_text", uid),
        }

    if fallback_pair_target is not None and role in {"subject", "object"}:
        prefix = "subject" if role == "subject" else "object"
        return {
            "uid": uid,
            "id": fallback_pair_target.get(f"{prefix}_id"),
            "type": fallback_pair_target.get(f"{prefix}_type"),
            "mention_text": uid,
        }

    return {
        "uid": uid,
        "id": None,
        "type": None,
        "mention_text": uid,
    }


def build_explicit_edges(paragraph):
    """
    Build explicit relation graph from paragraph["explicit_relations_in_text"].

    Key:
        (subject_uid, relation, object_uid)

    Value:
        explicit relation record
    """
    edges = {}
    for r in paragraph.get("explicit_relations_in_text", []):
        s = r.get("subject_uid")
        rel = r.get("relation")
        o = r.get("object_uid")
        if s and rel and o:
            edges[(s, rel, o)] = r
    return edges


def is_explicit_direct_or_inverse(A, R, C, explicit_edges):
    """
    Exclude if target pair is explicitly expressed as:
        A R C
    or inverse explicitly expressed as:
        C inverse(R) A
    """
    invR = INV.get(R)

    if STEP4T_EXCLUDE_EXPLICIT_DIRECT and (A, R, C) in explicit_edges:
        return True

    if STEP4T_EXCLUDE_EXPLICIT_INVERSE and invR and (C, invR, A) in explicit_edges:
        return True

    return False

In [5]:
# ============================================================
# Cell 4. Explicit support-chain rules
# ============================================================

def find_all_supporting_chains(A, R, C, object_uids, explicit_edges):
    """
    Find all explicit relation-chain support paths for target:
        A --R--> C

    Current supported rules:

    1. chain_same_direction
       A --R--> B
       B --R--> C
       => A --R--> C

    2. shared_anchor_opposite_sides
       A --R--> B
       C --inverse(R)--> B
       => A --R--> C

    3. anchor_between_reversed_surface_form
       B --inverse(R)--> A
       B --R--> C
       => A --R--> C
    """
    if R not in INV:
        return []

    invR = INV[R]

    # Symmetric relations are not suitable for this transitive setup.
    if invR == R:
        return []

    supports = []

    for B in object_uids:
        if B in {A, C}:
            continue

        # Rule 1
        if (
            "chain_same_direction" in STEP4T_SUPPORT_RULES
            and (A, R, B) in explicit_edges
            and (B, R, C) in explicit_edges
        ):
            supports.append({
                "implicit_rule": "chain_same_direction",
                "anchor_uid": B,
                "supporting_edges": [
                    explicit_edges[(A, R, B)],
                    explicit_edges[(B, R, C)],
                ],
            })

        # Rule 2
        if (
            "shared_anchor_opposite_sides" in STEP4T_SUPPORT_RULES
            and (A, R, B) in explicit_edges
            and (C, invR, B) in explicit_edges
        ):
            supports.append({
                "implicit_rule": "shared_anchor_opposite_sides",
                "anchor_uid": B,
                "supporting_edges": [
                    explicit_edges[(A, R, B)],
                    explicit_edges[(C, invR, B)],
                ],
            })

        # Rule 3
        if (
            "anchor_between_reversed_surface_form" in STEP4T_SUPPORT_RULES
            and (B, invR, A) in explicit_edges
            and (B, R, C) in explicit_edges
        ):
            supports.append({
                "implicit_rule": "anchor_between_reversed_surface_form",
                "anchor_uid": B,
                "supporting_edges": [
                    explicit_edges[(B, invR, A)],
                    explicit_edges[(B, R, C)],
                ],
            })

    # Only keep supports with enough explicit edges.
    supports = [
        s for s in supports
        if len(s.get("supporting_edges", [])) >= STEP4T_MIN_SUPPORT_EDGES
    ]

    return supports


def choose_primary_support(supports):
    """
    Choose one canonical support path.
    Current strategy:
      1. Prefer rule order defined below.
      2. Then lexical order of anchor uid.
    """
    if not supports:
        return None

    rule_priority = {
        "chain_same_direction": 0,
        "shared_anchor_opposite_sides": 1,
        "anchor_between_reversed_surface_form": 2,
    }

    return sorted(
        supports,
        key=lambda s: (
            rule_priority.get(s.get("implicit_rule"), 999),
            str(s.get("anchor_uid")),
        )
    )[0]

In [6]:
# ============================================================
# Cell 5. Mention span selection
# ============================================================

def find_span_in_supporting_sentences(text, uid, mention_text, supporting_edges):
    """
    Prefer the mention occurrence inside supporting relation sentences.
    Return paragraph-level span if found.
    """
    if not mention_text:
        mention_text = uid

    for edge in supporting_edges:
        sentence = edge.get("sentence", "")
        if not sentence:
            continue

        # Only search supporting sentence if this edge involves the uid.
        if edge.get("subject_uid") != uid and edge.get("object_uid") != uid:
            continue

        sent_start = text.find(sentence)
        if sent_start < 0:
            continue

        local_pos = sentence.find(mention_text)
        if local_pos >= 0:
            return {
                "start": sent_start + local_pos,
                "end": sent_start + local_pos + len(mention_text),
            }

    return None


def choose_probe_span(text, uid, mention_text, supporting_edges):
    """
    Choose one paragraph span for probe subject/object.

    Priority:
      1. occurrence inside supporting relation sentence
      2. first exact occurrence in paragraph
      3. None
    """
    all_spans = find_all_spans(text, mention_text)

    if STEP4T_SPAN_SELECTION_MODE == "prefer_supporting_sentence_then_first_occurrence":
        preferred = find_span_in_supporting_sentences(
            text=text,
            uid=uid,
            mention_text=mention_text,
            supporting_edges=supporting_edges,
        )
        if preferred is not None:
            return preferred, all_spans

    if all_spans:
        return all_spans[0], all_spans

    return None, all_spans

In [7]:
# ============================================================
# Cell 6. Build one Step4T sample
# ============================================================

def compact_support_edge(edge):
    """
    Store relevant information from explicit relation record.
    """
    return {
        "sentence": edge.get("sentence"),
        "template": edge.get("template"),
        "template_index": edge.get("template_index"),
        "generation_mode": edge.get("generation_mode"),
        "subject_uid": edge.get("subject_uid"),
        "subject_id": edge.get("subject_id"),
        "subject_type": edge.get("subject_type"),
        "relation": edge.get("relation"),
        "object_uid": edge.get("object_uid"),
        "object_id": edge.get("object_id"),
        "object_type": edge.get("object_type"),
        "was_swapped_for_text": edge.get("was_swapped_for_text"),
        "source_relation": edge.get("source_relation"),
    }


def make_step4T_sample(paragraph, pair_target, primary_support, all_supports, source_step3_file):
    text = paragraph.get("text", "")
    obj_index = build_object_index(paragraph)

    A = pair_target["subject_uid"]
    C = pair_target["object_uid"]
    R = pair_target["relation_label"]

    subj_meta = get_obj_meta(A, obj_index, pair_target, role="subject")
    obj_meta = get_obj_meta(C, obj_index, pair_target, role="object")

    primary_edges = primary_support["supporting_edges"]
    compact_primary_supports = [compact_support_edge(e) for e in primary_edges]

    if STEP4T_SAVE_ALL_SUPPORTING_PATHS:
        all_supporting_paths = []
        for support in all_supports:
            all_supporting_paths.append({
                "implicit_rule": support.get("implicit_rule"),
                "anchor_uid": support.get("anchor_uid"),
                "supporting_relations": [
                    compact_support_edge(e)
                    for e in support.get("supporting_edges", [])
                ],
            })
    else:
        all_supporting_paths = None

    subj_span, subj_all_spans = choose_probe_span(
        text=text,
        uid=A,
        mention_text=subj_meta["mention_text"],
        supporting_edges=primary_edges,
    )

    obj_span, obj_all_spans = choose_probe_span(
        text=text,
        uid=C,
        mention_text=obj_meta["mention_text"],
        supporting_edges=primary_edges,
    )

    scene = paragraph.get("scene") or pair_target.get("scene")
    paragraph_id = paragraph.get("paragraph_id") or pair_target.get("paragraph_id")
    chunk_id = paragraph.get("chunk_id") or pair_target.get("chunk_id")
    cluster_id = paragraph.get("cluster_id") or pair_target.get("cluster_id")

    sample_key = (
        f"{paragraph_id}|{A}|{C}|{R}|"
        f"{primary_support['implicit_rule']}|{primary_support['anchor_uid']}"
    )
    sample_id = "step4T_" + stable_hash(sample_key)
    pair_group_id = f"{paragraph_id}|{A}|{C}|{R}|{IMPLICIT_TRANSITIVE}"

    evidence = {
        "evidence_type": IMPLICIT_TRANSITIVE,
        "probe_direction_relative_to_text": "implicit",
        "has_explicit_evidence_sentence": False,
        "evidence_sentence": None,
        "sentence_index_in_paragraph": None,
        "surface_order": None,

        "supporting_relation_source": STEP4T_SUPPORTING_RELATION_SOURCE,
        "implicit_rule": primary_support["implicit_rule"],
        "anchor_uid": primary_support["anchor_uid"],
        "supporting_relations": compact_primary_supports,

        "num_supporting_paths": len(all_supports),
        "original_pair_evidence_type": pair_target.get("pair_evidence_type"),
        "label_source": STEP4T_LABEL_SOURCE,
    }

    if STEP4T_SAVE_ALL_SUPPORTING_PATHS:
        evidence["all_supporting_paths"] = all_supporting_paths

    sample = {
        "sample_id": sample_id,
        "pair_group_id": pair_group_id,

        "scene": scene,
        "experiment": STEP4T_EXPERIMENT_NAME,
        "generation_mode": paragraph.get("generation_mode"),
        "paragraph_id": paragraph_id,
        "chunk_id": chunk_id,
        "cluster_id": cluster_id,
        "grid_i": paragraph.get("grid_i", pair_target.get("grid_i")),
        "grid_j": paragraph.get("grid_j", pair_target.get("grid_j")),
        "paragraph_index_within_cluster": paragraph.get("paragraph_index_within_cluster"),

        "text": text,

        "evidence": evidence,

        "probe_pair": {
            "probe_subject_uid": A,
            "probe_subject_id": subj_meta["id"],
            "probe_subject_type": subj_meta["type"],
            "probe_subject_mention_text": subj_meta["mention_text"],
            "probe_subject_all_char_spans_in_paragraph": subj_all_spans,
            "probe_subject_span_in_evidence_sentence": None,
            "probe_subject_span_in_paragraph": subj_span,

            "probe_object_uid": C,
            "probe_object_id": obj_meta["id"],
            "probe_object_type": obj_meta["type"],
            "probe_object_mention_text": obj_meta["mention_text"],
            "probe_object_all_char_spans_in_paragraph": obj_all_spans,
            "probe_object_span_in_evidence_sentence": None,
            "probe_object_span_in_paragraph": obj_span,

            "probe_relation_label": R,
            "label_space": "directed_spatial_relation",
            "has_relation_label": True,

            "is_explicit_in_text": False,
            "is_inverse_of_text_relation": False,
            "is_symmetric_relation": (R == "near"),
            "is_directional_relation": (R not in {"near", "none"}),
            "is_implicit_transitive": True,
        },

        "source_relation": {
            "relation_source": pair_target.get("relation_source"),
            "source_relation_summary": pair_target.get("source_relation_summary"),
            "label_source": STEP4T_LABEL_SOURCE,
            "sample_source": STEP4T_SAMPLE_SOURCE,
            "text_source": STEP4T_TEXT_SOURCE,
            "original_pair_target": {
                "subject_uid": pair_target.get("subject_uid"),
                "subject_id": pair_target.get("subject_id"),
                "subject_type": pair_target.get("subject_type"),
                "relation_label": pair_target.get("relation_label"),
                "object_uid": pair_target.get("object_uid"),
                "object_id": pair_target.get("object_id"),
                "object_type": pair_target.get("object_type"),
                "pair_evidence_type": pair_target.get("pair_evidence_type"),
                "relation_source": pair_target.get("relation_source"),
            },
        },

        "geometry": {
            "has_geometry_label": pair_target.get("geometry_label") is not None,
            "geometry_label": pair_target.get("geometry_label"),
        },

        "source_step3_file": source_step3_file,
        "source_step3_scene": scene,
    }

    return sample

In [8]:
# ============================================================
# Cell 7. Construct samples from one paragraph
# ============================================================

def construct_from_paragraph(paragraph, source_step3_file):
    explicit_edges = build_explicit_edges(paragraph)

    obj_index = build_object_index(paragraph)
    object_uids = list(obj_index.keys())

    samples = []
    debug_counts = Counter()

    pair_targets = paragraph.get("pair_targets", [])

    for p in pair_targets:
        debug_counts["pair_targets_total"] += 1

        if p.get("pair_evidence_type") != STEP4T_CANDIDATE_EVIDENCE_TYPE:
            continue
        debug_counts["candidate_implicit_labeled"] += 1

        if not p.get("has_relation_label", False):
            continue
        debug_counts["candidate_has_label"] += 1

        A = p.get("subject_uid")
        C = p.get("object_uid")
        R = p.get("relation_label")

        if not A or not C or not R:
            debug_counts["filtered_missing_target_fields"] += 1
            continue

        if R not in ALLOWED_LABELS:
            debug_counts["filtered_label_not_allowed"] += 1
            continue
        debug_counts["candidate_allowed_label"] += 1

        if is_explicit_direct_or_inverse(A, R, C, explicit_edges):
            debug_counts["filtered_explicit_direct_or_inverse"] += 1
            continue

        all_supports = find_all_supporting_chains(
            A=A,
            R=R,
            C=C,
            object_uids=object_uids,
            explicit_edges=explicit_edges,
        )

        if not all_supports:
            debug_counts["filtered_no_support_chain"] += 1
            continue

        primary_support = choose_primary_support(all_supports)

        if primary_support is None:
            debug_counts["filtered_no_primary_support"] += 1
            continue

        sample = make_step4T_sample(
            paragraph=paragraph,
            pair_target=p,
            primary_support=primary_support,
            all_supports=all_supports,
            source_step3_file=source_step3_file,
        )

        # Optional: if main output requires valid spans, filter here.
        if STEP4T_REQUIRE_VALID_PROBE_SPANS_FOR_MAIN_OUTPUT:
            subj_span = sample["probe_pair"]["probe_subject_span_in_paragraph"]
            obj_span = sample["probe_pair"]["probe_object_span_in_paragraph"]
            if subj_span is None or obj_span is None:
                debug_counts["filtered_missing_probe_span_for_main_output"] += 1
                continue

        samples.append(sample)

        debug_counts["kept_implicit_transitive"] += 1
        debug_counts[f"kept_label::{R}"] += 1
        debug_counts[f"kept_rule::{primary_support['implicit_rule']}"] += 1

    return samples, debug_counts

In [9]:
# ============================================================
# Cell 8. Run construction for all Step3 files
# ============================================================

all_samples = []
global_counts = Counter()
per_file_counts = {}

for fname, data in step3_data_by_file.items():
    paragraphs = data.get("paragraphs", [])
    file_samples = []
    file_counts = Counter()

    for para in paragraphs:
        samples, counts = construct_from_paragraph(
            paragraph=para,
            source_step3_file=fname,
        )
        file_samples.extend(samples)
        file_counts.update(counts)

    scene = data.get("scene") or fname.replace("step3_", "").split("_")[0]

    per_scene_output = OUT_DIR / f"step4T_{scene}_implicit_transitive_probe_samples.json"
    with open(per_scene_output, "w", encoding="utf-8") as f:
        json.dump(file_samples, f, ensure_ascii=False, indent=2)

    if STEP4T_PRINT_FILE_SUMMARY:
        print(f"\nSaved {len(file_samples)} samples -> {per_scene_output}")
        print("Counts:")
        for k, v in file_counts.most_common():
            print(f"  {k}: {v}")

    all_samples.extend(file_samples)
    global_counts.update(file_counts)
    per_file_counts[fname] = dict(file_counts)

combined_path = OUT_DIR / STEP4T_COMBINED_OUTPUT_FILE
with open(combined_path, "w", encoding="utf-8") as f:
    json.dump(all_samples, f, ensure_ascii=False, indent=2)

print("\n==============================")
print("ALL DONE")
print("==============================")
print("Total implicit_transitive samples:", len(all_samples))
print("Combined output:", combined_path)

if STEP4T_PRINT_GLOBAL_SUMMARY:
    print("\nGlobal counts:")
    for k, v in global_counts.most_common():
        print(f"  {k}: {v}")


Saved 30 samples -> /content/pilot_4/data/p4_expB_settingA1_lr/step4T_FloorPlan1_implicit_transitive_probe_samples.json
Counts:
  pair_targets_total: 2666
  candidate_implicit_labeled: 1018
  candidate_has_label: 1018
  candidate_allowed_label: 576
  filtered_no_support_chain: 546
  filtered_label_not_allowed: 442
  kept_implicit_transitive: 30
  kept_label::left_of: 17
  kept_label::right_of: 13
  kept_rule::chain_same_direction: 10
  kept_rule::shared_anchor_opposite_sides: 10
  kept_rule::anchor_between_reversed_surface_form: 10

Saved 23 samples -> /content/pilot_4/data/p4_expB_settingA1_lr/step4T_FloorPlan2_implicit_transitive_probe_samples.json
Counts:
  pair_targets_total: 2074
  candidate_implicit_labeled: 760
  candidate_has_label: 760
  candidate_allowed_label: 432
  filtered_no_support_chain: 409
  filtered_label_not_allowed: 328
  kept_implicit_transitive: 23
  kept_label::right_of: 12
  kept_label::left_of: 11
  kept_rule::chain_same_direction: 9
  kept_rule::anchor_betwe

In [10]:
# ============================================================
# Cell 9. Save valid-span output
# ============================================================

valid_span_samples = []

for s in all_samples:
    subj_span = s["probe_pair"]["probe_subject_span_in_paragraph"]
    obj_span = s["probe_pair"]["probe_object_span_in_paragraph"]

    if subj_span is not None and obj_span is not None:
        valid_span_samples.append(s)

if STEP4T_SAVE_VALID_SPANS_FILE:
    valid_path = OUT_DIR / STEP4T_VALID_SPANS_OUTPUT_FILE
    with open(valid_path, "w", encoding="utf-8") as f:
        json.dump(valid_span_samples, f, ensure_ascii=False, indent=2)

    print("Original samples:", len(all_samples))
    print("Valid-span samples:", len(valid_span_samples))
    print("Saved valid-span file:", valid_path)
else:
    print("STEP4T_SAVE_VALID_SPANS_FILE is False; skipped valid-span output.")

Original samples: 167
Valid-span samples: 167
Saved valid-span file: /content/pilot_4/data/p4_expB_settingA1_lr/p4_expB_settingA1_implicit_transitive_lr_spans.json


In [11]:
# ============================================================
# Cell 10. Save summary
# ============================================================

label_counts = Counter()
scene_counts = Counter()
rule_counts = Counter()
cluster_counts = Counter()
span_missing_counts = Counter()
support_path_counts = Counter()

for s in all_samples:
    label = s["probe_pair"]["probe_relation_label"]
    scene = s["scene"]
    rule = s["evidence"]["implicit_rule"]
    cluster_id = s.get("cluster_id")
    num_paths = s["evidence"].get("num_supporting_paths", 1)

    label_counts[label] += 1
    scene_counts[scene] += 1
    rule_counts[rule] += 1
    cluster_counts[cluster_id] += 1
    support_path_counts[num_paths] += 1

    if s["probe_pair"]["probe_subject_span_in_paragraph"] is None:
        span_missing_counts["subject_span_missing"] += 1
    if s["probe_pair"]["probe_object_span_in_paragraph"] is None:
        span_missing_counts["object_span_missing"] += 1

summary = {
    "experiment": STEP4T_EXPERIMENT_NAME,
    "definition": STEP4T_DEFINITION,
    "description": (
        "Construct implicit_transitive probe samples from Step3 outputs. "
        "Candidates are Step3 implicit_labeled pair_targets. "
        "Kept samples must be supported by explicit relation chains "
        "inside the same paragraph."
    ),
    "input_dir": str(STEP3_DIR),
    "input_pattern": STEP4T_INPUT_PATTERN,
    "output_dir": str(OUT_DIR),
    "combined_output_file": STEP4T_COMBINED_OUTPUT_FILE,
    "valid_spans_output_file": STEP4T_VALID_SPANS_OUTPUT_FILE,
    "allowed_labels": sorted(ALLOWED_LABELS),
    "candidate_evidence_type": STEP4T_CANDIDATE_EVIDENCE_TYPE,
    "new_evidence_type": IMPLICIT_TRANSITIVE,
    "support_rules": sorted(STEP4T_SUPPORT_RULES),
    "num_samples": len(all_samples),
    "num_valid_span_samples": len(valid_span_samples),
    "label_counts": dict(label_counts),
    "scene_counts": dict(scene_counts),
    "rule_counts": dict(rule_counts),
    "cluster_counts": dict(cluster_counts),
    "span_missing_counts": dict(span_missing_counts),
    "support_path_counts": dict(support_path_counts),
    "global_construction_counts": dict(global_counts),
    "per_file_counts": per_file_counts,
}

summary_path = OUT_DIR / STEP4T_SUMMARY_FILE
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved summary:", summary_path)
print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])

Saved summary: /content/pilot_4/data/p4_expB_settingA1_lr/p4_expB_settingA1_lr_output_summary.json
{
  "experiment": "Step4_expB_settingA1_implicit_transitive_lr",
  "definition": {
    "name": "implicit_transitive",
    "formal_definition": "implicit_transitive = implicit_labeled AND not explicit_direct_or_inverse AND explicit_relation_chain_supported",
    "candidate_condition": "pair_target.pair_evidence_type == implicit_labeled",
    "label_source": "step3_pair_targets_from_step2_geometry_or_topology",
    "support_source": "paragraph_explicit_relations",
    "text_intervention": "none; use original Step3 paragraph text",
    "new_text_generated": false
  },
  "description": "Construct implicit_transitive probe samples from Step3 outputs. Candidates are Step3 implicit_labeled pair_targets. Kept samples must be supported by explicit relation chains inside the same paragraph.",
  "input_dir": "/content/pilot_4/data/step3A_diverse_text_preserve_text_direction",
  "input_pattern": "Flo

In [12]:
# ============================================================
# Cell 11. Export CSV index
# ============================================================

rows = []

for s in all_samples:
    rows.append({
        "sample_id": s["sample_id"],
        "scene": s["scene"],
        "paragraph_id": s["paragraph_id"],
        "cluster_id": s["cluster_id"],
        "subject": s["probe_pair"]["probe_subject_uid"],
        "relation": s["probe_pair"]["probe_relation_label"],
        "object": s["probe_pair"]["probe_object_uid"],
        "rule": s["evidence"]["implicit_rule"],
        "anchor": s["evidence"]["anchor_uid"],
        "num_supporting_paths": s["evidence"].get("num_supporting_paths", 1),
        "subject_span_ok": s["probe_pair"]["probe_subject_span_in_paragraph"] is not None,
        "object_span_ok": s["probe_pair"]["probe_object_span_in_paragraph"] is not None,
        "source_step3_file": s["source_step3_file"],
    })

df = pd.DataFrame(rows)

if STEP4T_SAVE_INDEX_CSV:
    csv_path = OUT_DIR / STEP4T_INDEX_CSV_FILE
    df.to_csv(csv_path, index=False)
    print("CSV saved:", csv_path)
else:
    print("STEP4T_SAVE_INDEX_CSV is False; skipped CSV export.")

display(df.head(20))

if len(df) > 0:
    print("\nLabel counts:")
    display(df["relation"].value_counts())

    print("\nRule counts:")
    display(df["rule"].value_counts())

    print("\nScene counts:")
    display(df["scene"].value_counts())
else:
    print("No samples constructed.")

CSV saved: /content/pilot_4/data/p4_expB_settingA1_lr/p4_expB_settingA1_lr_output_index.csv


,sample_id,scene,paragraph_id,cluster_id,subject,relation,object,rule,anchor,num_supporting_paths,subject_span_ok,object_span_ok,source_step3_file
0,step4T_7cef4dc53f14655d8967,FloorPlan1,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_A_di...,FloorPlan1_chunk_0_0_cluster_0,soap_bottle_1,left_of,pot_1,chain_same_direction,potato_1,1,True,True,FloorPlan1_experiment_A_text_diverse.json
1,step4T_ce30cad906b3b778ae85,FloorPlan1,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_A_di...,FloorPlan1_chunk_0_0_cluster_0,soap_bottle_1,left_of,wine_bottle_1,shared_anchor_opposite_sides,potato_1,1,True,True,FloorPlan1_experiment_A_text_diverse.json
2,step4T_cb097e89bf866eab301d,FloorPlan1,FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_A_di...,FloorPlan1_chunk_0_0_cluster_0,wine_bottle_1,right_of,soap_bottle_1,shared_anchor_opposite_sides,potato_1,1,True,True,FloorPlan1_experiment_A_text_diverse.json
3,step4T_8131a4dbb707a0ee6f91,FloorPlan1,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_A_di...,FloorPlan1_chunk_1_0_cluster_0,cabinet_2,right_of,stove_knob_2,anchor_between_reversed_surface_form,pepper_shaker_1,1,True,True,FloorPlan1_experiment_A_text_diverse.json
4,step4T_45924177cfdc2c5f0cc1,FloorPlan1,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_A_di...,FloorPlan1_chunk_1_0_cluster_0,kettle_1,right_of,salt_shaker_1,chain_same_direction,pan_1,1,True,True,FloorPlan1_experiment_A_text_diverse.json
5,step4T_7bb373d3254542f99ffe,FloorPlan1,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_A_di...,FloorPlan1_chunk_1_0_cluster_0,kettle_1,right_of,stove_burner_1,chain_same_direction,pan_1,1,True,True,FloorPlan1_experiment_A_text_diverse.json
6,step4T_22c83ec872bebac94c83,FloorPlan1,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_A_di...,FloorPlan1_chunk_1_0_cluster_0,kettle_1,right_of,stove_knob_2,shared_anchor_opposite_sides,pan_1,2,True,True,FloorPlan1_experiment_A_text_diverse.json
7,step4T_67f1b760dbb8adc19649,FloorPlan1,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_A_di...,FloorPlan1_chunk_1_0_cluster_0,stove_burner_1,left_of,kettle_1,chain_same_direction,pepper_shaker_1,1,True,True,FloorPlan1_experiment_A_text_diverse.json
8,step4T_a3f961d30a56ad7d5538,FloorPlan1,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_A_di...,FloorPlan1_chunk_1_0_cluster_0,stove_knob_2,left_of,cabinet_2,anchor_between_reversed_surface_form,pepper_shaker_1,1,True,True,FloorPlan1_experiment_A_text_diverse.json
9,step4T_66c7724600fd43127b9c,FloorPlan1,FloorPlan1_FloorPlan1_chunk_1_0_cluster_0_A_di...,FloorPlan1_chunk_1_0_cluster_0,stove_knob_2,left_of,kettle_1,shared_anchor_opposite_sides,pan_1,2,True,True,FloorPlan1_experiment_A_text_diverse.json



Label counts:


,count
relation,
left_of,86
right_of,81



Rule counts:


,count
rule,
chain_same_direction,83
anchor_between_reversed_surface_form,47
shared_anchor_opposite_sides,37



Scene counts:


,count
scene,
FloorPlan4,40
FloorPlan5,31
FloorPlan3,30
FloorPlan1,30
FloorPlan2,23
FloorPlan6,13


In [13]:
# ============================================================
# Cell 12. Inspect examples
# ============================================================

if STEP4T_PRINT_EXAMPLES:
    N = min(STEP4T_NUM_EXAMPLES_TO_PRINT, len(all_samples))

    for i in range(N):
        s = all_samples[i]
        print("=" * 100)
        print("sample_id:", s["sample_id"])
        print("scene:", s["scene"])
        print("paragraph_id:", s["paragraph_id"])
        print("label:", s["probe_pair"]["probe_relation_label"])
        print("probe:", s["probe_pair"]["probe_subject_uid"], "->", s["probe_pair"]["probe_object_uid"])
        print("rule:", s["evidence"]["implicit_rule"])
        print("anchor:", s["evidence"]["anchor_uid"])
        print("num_supporting_paths:", s["evidence"].get("num_supporting_paths", 1))

        print("\nSupporting relations:")
        for r in s["evidence"]["supporting_relations"]:
            print("  -", r["subject_uid"], r["relation"], r["object_uid"])
            print("    sentence:", r["sentence"])

        print("\nProbe spans:")
        print("  subject span:", s["probe_pair"]["probe_subject_span_in_paragraph"])
        print("  object span:", s["probe_pair"]["probe_object_span_in_paragraph"])

        print("\nText preview:")
        print(s["text"][:900])
else:
    print("STEP4T_PRINT_EXAMPLES is False; skipped example printing.")

sample_id: step4T_7cef4dc53f14655d8967
scene: FloorPlan1
paragraph_id: FloorPlan1_FloorPlan1_chunk_0_0_cluster_0_A_diverse_0
label: left_of
probe: soap_bottle_1 -> pot_1
rule: chain_same_direction
anchor: potato_1
num_supporting_paths: 1

Supporting relations:
  - soap_bottle_1 left_of potato_1
    sentence: soap_bottle_1 is on the left side of potato_1.
  - potato_1 left_of pot_1
    sentence: potato_1 is positioned to the left of pot_1.

Probe spans:
  subject span: {'start': 213, 'end': 226}
  object span: {'start': 546, 'end': 551}

Text preview:
Within this local scene, On top of counter_top_1, there is coffee_machine_1. mug_1 is resting on counter_top_1. pot_1 is on counter_top_1. counter_top_1 sits above dish_sponge_1. cabinet_1 includes wine_bottle_1. soap_bottle_1 is on the left side of potato_1. cabinet_2 can be seen to the right of paper_towel_roll_1. Below wine_bottle_1, there is counter_top_1. pot_1 is below cabinet_1. cabinet_2 holds dish_sponge_1. dish_sponge_1 sits to t

In [14]:
# ============================================================
# Cell 13. Sanity checks
# ============================================================

def check_sample_basic_consistency(sample):
    errors = []

    if sample["evidence"]["evidence_type"] != IMPLICIT_TRANSITIVE:
        errors.append("wrong_evidence_type")

    label = sample["probe_pair"]["probe_relation_label"]
    if label not in ALLOWED_LABELS:
        errors.append("label_not_allowed")

    if not sample["probe_pair"]["is_implicit_transitive"]:
        errors.append("is_implicit_transitive_false")

    if sample["probe_pair"]["is_explicit_in_text"]:
        errors.append("is_explicit_in_text_true")

    if not sample["evidence"].get("supporting_relations"):
        errors.append("missing_supporting_relations")

    if sample["evidence"].get("num_supporting_paths", 0) < 1:
        errors.append("num_supporting_paths_lt_1")

    return errors


error_counter = Counter()
bad_examples = []

for s in all_samples:
    errs = check_sample_basic_consistency(s)
    for e in errs:
        error_counter[e] += 1
    if errs and len(bad_examples) < 5:
        bad_examples.append((s["sample_id"], errs))

print("Sanity check error counts:", dict(error_counter))

if bad_examples:
    print("\nBad examples:")
    for sample_id, errs in bad_examples:
        print(sample_id, errs)
else:
    print("All samples passed basic consistency checks.")

Sanity check error counts: {}
All samples passed basic consistency checks.


In [15]:
# Download results

output_dir = Path(cfg.STEP4T_OUTPUT_DIR)

if not output_dir.exists():
    raise FileNotFoundError(f"STEP4T_OUTPUT_DIR does not exist: {output_dir}")

output_files = [p for p in output_dir.rglob("*") if p.is_file()]

if not output_files:
    raise FileNotFoundError(f"No files found in STEP4T_OUTPUT_DIR: {output_dir}")

zip_base = Path("/content") / cfg.STEP4T_EXPERIMENT_NAME
zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(output_dir.parent),
    base_dir=output_dir.name,
)

print("Step4T output directory:", output_dir)
print("Number of files packaged:", len(output_files))
print("Created zip:", zip_path)

files.download(zip_path)

Step4T output directory: /content/pilot_4/data/p4_expB_settingA1_lr
Number of files packaged: 10
Created zip: /content/Step4_expB_settingA1_implicit_transitive_lr.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>